# sorted-computational-graph — worked example 2: Topological sort visits a shared leaf exactly once

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `sorted-computational-graph`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """A minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries an optional `.recipe`
    populated by wrap_forward_fn. `requires_grad` is set by the wrapper."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

When one leaf node is consumed by multiple intermediate nodes, the DFS with a `perm` set (keyed by `id()`) prevents visiting the shared leaf twice. Without deduplication, the leaf would appear multiple times in the topological order, corrupting gradient accumulation — each extra visit would double-count its gradient contribution.

## Worked solution

**Step 1 — Build a graph where leaf `a` is used twice.** We create `u = f(a)` and `v = g(a)`, then `w = h(u, v)`. The leaf `a` is shared.

**Step 2 — Run topological_sort from w.** The DFS will reach `a` through both the `u` path and the `v` path.

**Step 3 — Check deduplication.** The `perm` set ensures `a` is only appended once. Without it, `a` would appear twice in the result.

**Step 4 — Verify the backward-pass property.** In the reversed order, `w` is first, `u` and `v` come next (in some order), and `a` is last — appearing exactly once.

In [ ]:
import torch as t

t.manual_seed(0)

class FakeRecipe:
    def __init__(self, parents):
        self.parents = {i: p for i, p in enumerate(parents)}

class FakeTensor:
    def __init__(self, name, parents=None):
        self.name = name
        self.recipe = FakeRecipe(parents) if parents else None
    def __repr__(self):
        return f'T({self.name})'

def topological_sort(node, get_children):
    result = []
    perm = set()
    temp = set()
    def visit(cur):
        cid = id(cur)
        if cid in perm:
            return
        if cid in temp:
            raise ValueError('Cycle detected')
        temp.add(cid)
        for child in get_children(cur):
            visit(child)
        temp.discard(cid)
        perm.add(cid)
        result.append(cur)
    visit(node)
    return result

def get_parents(n):
    if n.recipe is None:
        return []
    return list(n.recipe.parents.values())

# Shared leaf: a
a = FakeTensor('a')
u = FakeTensor('u', [a])       # u = f(a)
v = FakeTensor('v', [a])       # v = g(a)
w = FakeTensor('w', [u, v])    # w = h(u, v)

bwd_order = topological_sort(w, get_parents)[::-1]
names = [n.name for n in bwd_order]

print('Backward order:', names)
print('Length:', len(bwd_order), '(should be 4 — no duplicates)')
assert len(bwd_order) == 4, f'Expected 4 unique nodes, got {len(bwd_order)}'
assert bwd_order[0] is w
assert bwd_order[-1] is a, 'shared leaf should appear exactly once, at the end'